# **Demo: Structured Prompting and Engineering Tasks (Groq API)**

### **Steps to Follow:**
- **Step 1:** Set up the Groq API Client and Enter Your API Key
- **Step 2:** Define a Function to Get Completion
- **Step 3:** Implement the Four-Element Structured Framework
- **Step 4:** Execute Two Zero-Shot Engineering Tasks
- **Step 5:** Execute a Few-Shot Engineering Task

### **Step 1: Set up Groq API Client**
- Installs the `groq` Python package.
- Enter your key directly in the code cell (do not publish or share a populated key).
- Calls Groq's hosted `llama-3.3-70b-versatile` model; a local GPU is not required.

In [20]:
# Step 1: Install and configure the Groq API client
%pip install -q groq
import os
from dotenv import load_dotenv
from groq import Groq

# Step 1: Load variables from .env file
load_dotenv()

# Step 2: Read the Groq API key
api_key = os.getenv("GROQ_API_KEY")

# Step 3: Validate the key
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")

# Step 4: Initialize Groq client
client = Groq(api_key=api_key)

MODEL_NAME = "openai/gpt-oss-20b"

print(f"Groq client ready. Model: {MODEL_NAME}")


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Groq client ready. Model: openai/gpt-oss-20b


### **Step 2: Define a Function to Get Completion**
- Sends instruction prompts to the Groq chat completions endpoint.
- Sets `temperature=0` for more consistent outputs (exact reproducibility is not guaranteed).

In [21]:
# Step 2: Reusable completion helper using the Groq Chat Completions API
def get_completion(prompt, max_tokens=150):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_completion_tokens=max_tokens,
    )
    return (response.choices[0].message.content or "").strip()

print("✅ Groq completion helper ready.")

✅ Groq completion helper ready.


### **Step 3: Implement the Four-Element Structured Framework**
Structured prompts decouple core operational components to ensure clear boundaries:
1. **Instruction:** The primary imperative action.
2. **Context:** System background and technical constraints.
3. **Input Data:** The code snippet, data, or log to process.
4. **Output Indicator:** Format constraints and size boundaries.

In [22]:
# Step 3: Four-element prompt builder
def build_structured_prompt(instruction, context, input_data, output_indicator):
    return (
        f"Instruction: {instruction}\n"
        f"Context: {context}\n"
        f"Input: {input_data}\n"
        f"Output: {output_indicator}"
    )

print("✅ Structured prompt builder defined.")

✅ Structured prompt builder defined.


### **Step 4: Execute Two Zero-Shot Engineering Tasks**
Zero-shot prompting tests model comprehension relying strictly on explicit instructions and boundaries without demonstration exemplars:
- **Task A (Edge Case Detection):** Spotting potential runtime exceptions (`ZeroDivisionError`).
- **Task B (Security Remediation):** Refactoring string concatenation in SQL queries into parameterized calls (OWASP A03 Injection).

In [23]:
# --- Task A: Edge Case Detection ---
task_a_prompt = build_structured_prompt(
    instruction="Identify the bug in this function.",
    context="Calculates the average of a list of numbers.",
    input_data="def avg(nums):\n    return sum(nums) / len(nums)",
    output_indicator="Name the breaking edge case in exactly one sentence."
)

print("--- Zero-Shot Task A: Edge Case Detection ---")
print(get_completion(task_a_prompt, max_tokens=1024))

# --- Task B: Security Remediation (SQL Injection) ---
task_b_prompt = build_structured_prompt(
    instruction="Fix the SQL injection vulnerability.",
    context="Database lookup query.",
    input_data='query = f"SELECT * FROM users WHERE name = \'{name}\'"',
    output_indicator="Return only the corrected line of code using a parameterized query."
)

print("\n--- Zero-Shot Task B: Security Remediation ---")
print(get_completion(task_b_prompt, max_tokens=1024))

--- Zero-Shot Task A: Edge Case Detection ---
The function fails when given an empty list, causing a division‑by‑zero error.

--- Zero-Shot Task B: Security Remediation ---
query = "SELECT * FROM users WHERE name = %s"


### **Step 5: Execute a Few-Shot Engineering Task**
Few-shot prompting provides domain-specific demonstrations to establish an exact output format and decision logic:
- **Task C (Automated Defect Triage):** Evaluates incoming feedback, tags the corresponding engineering action (`NONE`, `CREATE_DEFECT`, `UPDATE_DOCS`), and assigns an engineering priority (`P0_BLOCKER` to `P3_LOW`).

In [24]:
# Step 5: Few-shot prompt for engineering defect triage
few_shot_triage_prompt = """
Classify the engineering action and priority based on customer feedback.

Example 1:
Feedback: "Great battery life, works as expected."
Action: NONE | Priority: P3_LOW

Example 2:
Feedback: "The application crashes immediately upon login."
Action: CREATE_DEFECT | Priority: P0_BLOCKER

Example 3:
Feedback: "The user manual has outdated screenshots for setup."
Action: UPDATE_DOCS | Priority: P2_MEDIUM

Current Task:
Feedback: "The item arrived early, but the instructions were missing."
Action:"""

print("--- Few-Shot Task C: Automated Triage ---")
print(get_completion(few_shot_triage_prompt, max_tokens=1024))

--- Few-Shot Task C: Automated Triage ---
Action: UPDATE_DOCS | Priority: P2_MEDIUM
